# Procesamiento ETL 2022 - Estadísticas Vitales

Este notebook se encarga de importar, limpiar, manejar nulos, normalizar y guardar de forma eficiente (en formato Parquet) los datasets correspondientes al año 2022 para:
- Nacimientos
- Muertes Fetales
- Muertes No Fetales

In [1]:
import os
import sys
import pandas as pd

# Agregar la carpeta principal al path para poder importar módulos propios
sys.path.append(os.path.abspath('..'))

from etl.cleaning import *
from etl.nulls import *
from etl.normalization import *
from etl.schema import *

# Asegurarnos de que directorios de salida existan
os.makedirs('../data/processed/nacimientos', exist_ok=True)
os.makedirs('../data/processed/fetales', exist_ok=True)
os.makedirs('../data/processed/no_fetales', exist_ok=True)

# 1. Nacimientos 2022

### Carga del archivo

In [2]:
rutaDatos = '../data/raw/nac2022.csv'
try:
    df_nac = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nac.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

C:\Users\edwar\AppData\Local\Temp\ipykernel_329724\1776939518.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nac = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')


Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573625 entries, 0 to 573624
Data columns (total 39 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        573625 non-null  int64  
 1   COD_MUNIC       573625 non-null  int64  
 2   AREANAC         573625 non-null  int64  
 3   SIT_PARTO       573625 non-null  int64  
 4   OTRO_SIT        1153 non-null    object 
 5   SEXO            573625 non-null  int64  
 6   PESO_NAC        573625 non-null  int64  
 7   TALLA_NAC       573625 non-null  int64  
 8   ANO             573625 non-null  int64  
 9   MES             573625 non-null  int64  
 10  ATEN_PAR        573625 non-null  int64  
 11  T_GES           573625 non-null  int64  
 12  T_GES_AGRU_CIE  573625 non-null  int64  
 13  NUMCONSUL       573625 non-null  int64  
 14  TIPO_PARTO      573625 non-null  int64  
 15  MUL_PARTO       573625 non-null  int64  
 16  APGAR1          573625 non-null  i

### Eliminación de duplicados

In [3]:
encontrar_duplicados(df_nac)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 573625
Filas duplicadas (incluyendo repetidas): 1144
Duplicados únicos a eliminar: 584


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,OTRO_SIT,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,...,N_HIJOSV,FECHA_NACM,N_EMB,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,ULTCURPAD,PROFESION,TIPOFORMULARIO
380,54,1,1,1,NaN,2,4,4,2022,12,...,2,NaN,2,1,1.0,32,9,5,1.0,1
441,20,45,9,9,NaN,2,9,9,2022,9,...,99,NaN,99,9,9.0,53,99,99,NaN,3
445,44,560,9,9,NaN,2,9,9,2022,6,...,99,NaN,99,9,9.0,999,99,99,NaN,3
457,44,847,9,9,NaN,2,9,9,2022,6,...,99,NaN,99,9,9.0,999,99,99,NaN,3
499,20,570,9,9,NaN,1,9,9,2022,4,...,99,NaN,99,9,9.0,24,99,99,NaN,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570849,13,1,1,1,NaN,2,4,4,2022,3,...,2,NaN,1,2,2.0,29,7,2,1.0,1
571766,50,1,1,1,NaN,2,6,4,2022,7,...,8,18/03/2020,10,2,2.0,33,3,8,1.0,1
571833,13,1,1,1,NaN,2,4,4,2022,3,...,2,NaN,1,2,2.0,29,7,2,1.0,1
572148,70,1,1,1,NaN,2,3,4,2022,6,...,3,25/03/2010,2,2,2.0,38,7,2,1.0,1


In [4]:
df_nac = eliminar_duplicados(df_nac)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 584
Registros eliminados: 584
Total final: 573041


### Eliminación de columnas que no son necesarias para el estudio

In [5]:
cols_a_borrar_nac = ['OTRO_SIT','APGAR1','APGAR2','IDPERTET','ULTCURMAD','FECHA_NACM','N_EMB','ULTCURPAD', 'TIPOFORMULARIO', 'T_GES_AGRU_CIE']
df_nac = eliminar_columnas(df_nac, cols_a_borrar_nac)

Borradas 10 columnas de la lista especificada.


### Estandarización de nombres de columnas

In [6]:
## Estandarización de nombres de columnas
df_nac = estandarizar_nombres_columnas(df_nac)

### Nueva columna

In [7]:
df_nac["tipo_evento"] = "NAC"

### Manejo de valores nulos

In [8]:
## Manejo de valores nulos
resumen_nulos = analizar_nulos(df_nac)

===== ANÁLISIS DE NULOS =====
Total de registros: 573041
Columnas con nulos: 6



,nulos,porcentaje (%)
idclasadmi,40701,7.102633
codptore,6279,1.095733
codmunre,6279,1.095733
area_res,6279,1.095733
profesion,2077,0.362452
codpres,1970,0.343780


In [9]:
df_nac[df_nac["codptore"].isnull()].shape


(6279, 30)

In [10]:
df_nac[df_nac["codmunre"].isnull()].shape

(6279, 30)

In [11]:
df_nac[df_nac["area_res"].isnull()].shape

(6279, 30)

In [12]:
df_nac[df_nac["profesion"].isnull()].shape

(2077, 30)

In [13]:
df_nac = df_nac.dropna(subset=["codptore", "codmunre", "area_res", "profesion"])

In [14]:
estrategia = {
    "idclasadmi": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nac = manejar_nulos(df_nac, estrategia)

In [15]:
df_nac.isnull().sum()

cod_dpto       0
cod_munic      0
areanac        0
sit_parto      0
sexo           0
peso_nac       0
talla_nac      0
ano            0
mes            0
aten_par       0
t_ges          0
numconsul      0
tipo_parto     0
mul_parto      0
idhemoclas     0
idfactorrh     0
edad_madre     0
est_civm       0
niv_edum       0
codpres        0
codptore       0
codmunre       0
area_res       0
n_hijosv       0
seg_social     0
idclasadmi     0
edad_padre     0
niv_edup       0
profesion      0
tipo_evento    0
dtype: int64

### Ajuste de tipos de datos y guardado

In [16]:
## Ajustar tipos de datos
df_nac = ajustar_tipos_datos(df_nac)

### Verificación de columnas

In [17]:
VALID_COLUMNS_NACIMIENTOS = {
    col: str(df_nac[col].dtype) for col in VALID_COLUMNS_NACIMIENTOS
}

validar_schema(df_nac, VALID_COLUMNS_NACIMIENTOS)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 30


# 2. Muertes Fetales 2022

### Carga del archivo

In [18]:
rutaDatos = '../data/raw/fetal2022.csv'
try:
    df_fet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_fet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28237 entries, 0 to 28236
Data columns (total 44 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   COD_DPTO        28237 non-null  int64  
 1   COD_MUNIC       28237 non-null  int64  
 2   A_DEFUN         28237 non-null  int64  
 3   SIT_DEFUN       28237 non-null  int64  
 4   OTRSITIODE      67 non-null     object 
 5   TIPO_DEFUN      28237 non-null  int64  
 6   ANO             28237 non-null  int64  
 7   MES             28237 non-null  int64  
 8   HORA            11048 non-null  float64
 9   MINUTOS         11048 non-null  float64
 10  SEXO            28237 non-null  int64  
 11  CODPRES         28143 non-null  float64
 12  CODPTORE        27894 non-null  float64
 13  CODMUNRE        27894 non-null  float64
 14  AREA_RES        27895 non-null  float64
 15  SEG_SOCIAL      28237 non-null  int64  
 16  IDADMISALUD     25863 non-null  float64
 17  P_PMAN_IR

### Eliminación de duplicados

In [19]:
encontrar_duplicados(df_fet) # Encontrar filas duplicadas

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 28237
Filas duplicadas (incluyendo repetidas): 374
Duplicados únicos a eliminar: 190


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
4,23,1,1,1,NaN,1,2022,3,8.0,0.0,...,1.0,NaN,NaN,NaN,1,P95*P019 P015,P015,402,1,80
63,11,1,1,1,NaN,1,2022,2,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
130,70,1,1,1,NaN,1,2022,9,14.0,10.0,...,1.0,1.0,NaN,NaN,1,Q999/P015,Q999,613,1,88
148,11,1,1,1,NaN,1,2022,9,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
168,5,1,1,1,NaN,1,2022,7,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P014,P014,402,1,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27769,76,1,1,1,NaN,1,2022,11,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
27797,11,1,1,1,NaN,1,2022,9,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P964/P019,P964,406,1,86
27850,8,1,1,1,NaN,1,2022,10,NaN,NaN,...,1.0,1.0,NaN,NaN,1,P964/Q899*Q079,P964,406,1,86
28097,8,1,1,1,NaN,1,2022,4,14.0,49.0,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80


In [20]:
df_fet = eliminar_duplicados(df_fet) # Eliminar filas duplicadas

===== LIMPIEZA DE DUPLICADOS =====


Duplicados detectados: 190
Registros eliminados: 190
Total final: 28047


### Eliminación de columnas que no son necesarias para el estudio

In [21]:
cols_a_borrar_fet  = ['HORA','MINUTOS','OTRSITIODE','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER', 'T_GES_AGRU_CIE']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet)

Borradas 19 columnas de la lista especificada.


### Estandarización de nombres de columnas

In [22]:
df_fet = estandarizar_nombres_columnas(df_fet)

### Nueva columna

In [23]:
df_fet["tipo_evento"] = "FETAL"

### Manejo de valores nulos

In [24]:
resumen_nulos = analizar_nulos(df_fet)

===== ANÁLISIS DE NULOS =====
Total de registros: 28047
Columnas con nulos: 5



,nulos,porcentaje (%)
idadmisalud,2365,8.432274
codmunre,340,1.212251
codptore,340,1.212251
area_res,339,1.208685
peso_nac,4,0.014262


In [25]:
df_fet[df_fet["codmunre"].isnull()].shape

(340, 26)

In [26]:
df_fet[df_fet["area_res"].isnull()].shape

(339, 26)

In [27]:
df_fet[df_fet["codptore"].isnull()].shape

(340, 26)

In [28]:
# Eliminar nulos geográficos
df_fet = df_fet.dropna(subset=["codptore", "codmunre", "area_res"])

In [29]:
# Imputar otros nulos
estrategia = {
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_fet = manejar_nulos(df_fet, estrategia)

In [30]:
df_fet.isnull().sum()

cod_dpto       0
cod_munic      0
a_defun        0
sit_defun      0
tipo_defun     0
ano            0
mes            0
sexo           0
codptore       0
codmunre       0
area_res       0
seg_social     0
idadmisalud    0
mu_parto       0
t_parto        0
tipo_emb       0
t_ges          0
peso_nac       1
edad_madre     0
n_hijosv       0
n_hijosm       0
est_civm       0
niv_edum       0
asis_med       0
cau_homol      0
tipo_evento    0
dtype: int64

### Ajuste de tipos de datos

In [31]:
df_fet = ajustar_tipos_datos(df_fet)

### Validación de columnas

In [32]:
VALID_COLUMNS_FETALES = {
    col: str(df_fet[col].dtype) for col in VALID_COLUMNS_FETALES
}

validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 24

➕ Columnas extra (2): ['seg_social', 'idadmisalud']


In [33]:
# Si hay columnas extra (seg_social, idadmisalud) que no estarán disponibles
# en años posteriores, las eliminamos para mantener el schema consistente
cols_extra_fet = [col for col in ['seg_social', 'idadmisalud'] if col in df_fet.columns]
if cols_extra_fet:
    df_fet = eliminar_columnas(df_fet, cols_extra_fet)

Borradas 2 columnas de la lista especificada.


In [34]:
validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 24


# 3. Muertes No Fetales 2022

### Carga del archivo

In [35]:
rutaDatos = '../data/raw/nofetal2022.csv'
try:
    df_nofet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nofet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287251 entries, 0 to 287250
Data columns (total 57 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        287251 non-null  int64  
 1   COD_MUNIC       287251 non-null  int64  
 2   A_DEFUN         287251 non-null  int64  
 3   SIT_DEFUN       287251 non-null  int64  
 4   OTRSITIODE      8027 non-null    object 
 5   TIPO_DEFUN      287251 non-null  int64  
 6   ANO             287251 non-null  int64  
 7   MES             287251 non-null  int64  
 8   HORA            282469 non-null  float64
 9   MINUTOS         282469 non-null  float64
 10  SEXO            287251 non-null  int64  
 11  EST_CIVIL       287251 non-null  int64  
 12  GRU_ED1         287251 non-null  int64  
 13  GRU_ED2         287251 non-null  int64  
 14  NIVEL_EDU       287251 non-null  int64  
 15  ULTCURFAL       287251 non-null  int64  
 16  MUERTEPORO      278891 non-null  f

### Eliminación de duplicados

In [36]:
encontrar_duplicados(df_nofet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 287251
Filas duplicadas (incluyendo repetidas): 179
Duplicados únicos a eliminar: 101


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL,TIPOFORMULARIO
7578,5,628,2,6,NSNR - SIN INFORMACIÓN,2,2022,5,0.0,0.0,...,NaN,NaN,NaN,2,T149/T149 Y34,Y348,513,1,102,1
10915,76,1,1,1,NaN,2,2022,4,14.0,30.0,...,NaN,NaN,NaN,1,I219/I509/I10,I219,303,1,51,1
19788,11,1,1,6,PLANTA DE ENERGIA TERMICA HIDRICA COMBUSTIBLE,2,2022,5,0.0,0.0,...,NaN,NaN,NaN,2,I490/T754/T754 W87,W878,507,1,97,1
20015,5,1,1,3,NaN,2,2022,4,5.0,0.0,...,NaN,1.0,NaN,1,R570/I10,I10,302,1,50,1
20066,5,1,1,3,NaN,2,2022,5,9.0,0.0,...,NaN,1.0,NaN,1,R570/I10,I10,302,1,50,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280118,19,743,3,3,NaN,2,2022,2,NaN,NaN,...,NaN,1.0,NaN,9,R98,R98,0,5,89,1
281570,5,79,9,9,NaN,2,2022,6,0.0,0.0,...,NaN,NaN,NaN,2,T71/T71/T71 X91,X919,512,1,101,1
284012,86,568,1,6,NSNR - SIN INFORMACIÓN,2,2022,9,0.0,0.0,...,NaN,NaN,NaN,2,T794/S062/T019 X95,X958,512,1,101,1
284217,19,743,3,3,NaN,2,2022,2,NaN,NaN,...,NaN,1.0,NaN,9,R98,R98,0,5,89,1


In [37]:
df_nofet = eliminar_duplicados(df_nofet)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 101
Registros eliminados: 101
Total final: 287150


### Eliminación de columnas que no son necesarias para el estudio

In [38]:
cols_a_borrar_nofet = ['OCUPACION','HORA','MINUTOS','OTRSITIODE','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE', 'CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER','MU_PARTO','CAU_HOMOL', 'T_GES_AGRU_CIE']
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_nofet)

Borradas 22 columnas de la lista especificada.


### Estandarización de nombres de columnas

In [39]:
df_nofet = estandarizar_nombres_columnas(df_nofet)

### Nueva columna

In [40]:
df_nofet["tipo_evento"] = "NOFETAL"

### Manejo de valores nulos

In [41]:
resumen_nulos = analizar_nulos(df_nofet)

===== ANÁLISIS DE NULOS =====
Total de registros: 287150
Columnas con nulos: 18



,nulos,porcentaje (%)
simuertepo,286867,99.901445
peso_nac,280492,97.681351
n_hijosm,280463,97.671252
edad_madre,280463,97.671252
est_civm,280463,97.671252
niv_edum,280463,97.671252
t_parto,280463,97.671252
n_hijosv,280463,97.671252
tipo_emb,280463,97.671252
t_ges,280463,97.671252


In [42]:
# Eliminar nulos geográficos (críticos para el análisis espacial)
df_nofet = df_nofet.dropna(subset=["codptore", "codmunre", "area_res"])

In [43]:
# Imputar otros nulos
estrategia = {
    "ocupacion": {"metodo": "fill", "valor": "DESCONOCIDO"},
    "muerteporo": {"metodo": "fill_mode"},
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nofet = manejar_nulos(df_nofet, estrategia)

In [44]:
# Eliminar columnas con nulos muy altos (>90%) que corresponden a datos
# de gestantes dentro de la base de no fetales (no aplican a la mayoría)
columnas_alta_nulidad = [
    "simuertepo",
    "t_ges",
    "niv_edum",
    "n_hijosm",
    "tipo_emb",
    "t_parto",
    "est_civm",
    "peso_nac",
    "edad_madre",
    "n_hijosv",
    "emb_mes",
    "emb_sem",
    "emb_fal"
]

df_nofet = eliminar_columnas(df_nofet, columnas_alta_nulidad)

Borradas 13 columnas de la lista especificada.


In [45]:
df_nofet.isnull().sum()

cod_dpto          0
cod_munic         0
a_defun           0
sit_defun         0
tipo_defun        0
ano               0
mes               0
sexo              0
est_civil         0
gru_ed1           0
gru_ed2           0
nivel_edu         0
ultcurfal         0
muerteporo        0
idpertet          0
codptore          0
codmunre          0
area_res          0
seg_social        0
idadmisalud       0
asis_med          0
tipoformulario    0
tipo_evento       0
dtype: int64

### Ajuste de tipos de datos 

In [46]:
df_nofet = ajustar_tipos_datos(df_nofet)

### Validación de columnas

In [47]:
VALID_COLUMNS_NO_FETALES = {
    col: str(df_nofet[col].dtype) for col in VALID_COLUMNS_NO_FETALES
}
validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 21

➕ Columnas extra (2): ['idadmisalud', 'tipoformulario']


In [48]:
cols_a_borrar_no_fet_extra = ['idadmisalud', "tipoformulario"]
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_no_fet_extra)

Borradas 2 columnas de la lista especificada.


In [49]:
validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 21
